In [82]:
import torchvision.models as models

base: models.ResNet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/aigoncharov/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:02<00:00, 23.1MB/s]


In [83]:
base

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [84]:
import torch
from torch import nn

model: models.ResNet = models.resnet18()

model.conv1 = nn.Conv2d(
    in_channels=127,
    out_channels=64,
    kernel_size=(3, 7),  # thin but long
    stride=(1, 2),  # no vertical shrink
    padding=(1, 3),
    bias=False,
)

model.maxpool = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 2), padding=(0, 1))

for name, m in model.named_modules():
    # Remove vertical downsampling
    if isinstance(m, nn.Conv2d) and m.stride == (2, 2):
        m.stride = (1, 2)

model.fc = nn.Linear(512, 2, bias=True)

model

ResNet(
  (conv1): Conv2d(127, 64, kernel_size=(3, 7), stride=(1, 2), padding=(1, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=(1, 3), stride=(1, 2), padding=(0, 1), dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU

In [85]:
import math


def transplant_conv1_weights(new_conv: torch.nn.Conv2d, old_conv: torch.nn.Conv2d, mode: str = "central_rows"):
    """
    Copy ImageNet 7×7 weights into a re-shaped conv1
    (arbitrary in_channels, arbitrary new kernel height ≤7).

    Args
    ----
    new_conv : nn.Conv2d  # e.g. (64, 127, 3, 7)
    old_conv : nn.Conv2d  # pretrained (64, 3, 7, 7)
    mode     : "central_rows" | "avg_height"
               central_rows → take rows 2-4 (3 rows) from 7×7
               avg_height   → mean along height, then repeat kH times
    """
    W = old_conv.weight.detach()  # (64, 3, 7, 7)
    kH_new, kW_new = new_conv.kernel_size  # (3, 7) in your case

    # 1️⃣  Crop / squeeze height ---------------------------------------------
    if mode == "central_rows":
        start = (W.size(2) - kH_new) // 2  # centre crop
        W = W[:, :, start : start + kH_new, :]  # (64, 3, kH_new, 7)
    elif mode == "avg_height":
        W = W.mean(dim=2, keepdim=True)  # (64, 3, 1, 7)
        W = W.expand(-1, -1, kH_new, -1).clone()  # (64, 3, kH_new, 7)
    else:
        raise ValueError("mode must be 'central_rows' or 'avg_height'")

    # 2️⃣  Collapse the 3 RGB filters into *one* template (optional but handy)
    W = W.mean(dim=1, keepdim=True)  # (64, 1, kH_new, 7)

    # 3️⃣  Tile / truncate that template to match the new #input channels -----
    Cin_new = new_conv.in_channels
    reps = math.ceil(Cin_new / 1)  # we have 1 channel in W now
    W = W.repeat(1, reps, 1, 1)[:, :Cin_new]  # (64, Cin_new, kH_new, 7)

    # 4️⃣  Variance-preserving rescale (optional but recommended)
    W /= Cin_new**0.5  # keeps fan-in unchanged

    # 5️⃣  Copy into the new conv
    with torch.no_grad():
        new_conv.weight.copy_(W.to(new_conv.weight.device))

In [86]:
transplant_conv1_weights(model.conv1, base.conv1, mode="central_rows")

for n, p in model.named_parameters():
    if n.startswith("conv1."):  # already patched
        continue
    if n.startswith("fc."):  # final layer
        continue
    with torch.no_grad():
        p.copy_(dict(base.named_parameters())[n])

del base
torch.cuda.empty_cache()

In [87]:
import gc
import os
import warnings

import mne

warnings.filterwarnings("ignore")

files = [x for x in os.listdir("./data/out") if "TI" in x]
persons = [x[:2] for x in files]

labels = []
for p in persons:
    epoch = mne.read_epochs(f"./data/source/{p}_TI_epochsICA.fif", verbose=False)
    labels.append(epoch.events[:, -1] - 2)

gc.collect()

assert len(persons) == len(labels)

persons, len(persons)

(['AL',
  'EI',
  'DK',
  'DS',
  'AB',
  'MK',
  'NM',
  'SG',
  'FG',
  'AR',
  'OK',
  'AA',
  'AM',
  'KS',
  'NB',
  'LJ',
  'IM',
  'NH',
  'AS',
  'ET',
  'AI',
  'DT'],
 22)

In [88]:
train_persons = persons[:18]
test_persons = persons[18:]

In [89]:
labels[0].shape

(60,)

In [90]:
import torch

DEVICE = "cpu"
if torch.mps.is_available():
    DEVICE = "mps"
if torch.cuda.is_available():
    DEVICE = "cuda"

DEVICE

'mps'

In [91]:
model.to(DEVICE)

ResNet(
  (conv1): Conv2d(127, 64, kernel_size=(3, 7), stride=(1, 2), padding=(1, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=(1, 3), stride=(1, 2), padding=(0, 1), dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU

In [92]:
import numpy as np
from torch.utils.data import Dataset


class NumpyDataset(Dataset):
    def __init__(self, folder: str, persons: list[str], labels):
        self.folder = folder
        self.persons = persons

        self.labels = labels

        self.current_chunk = None
        self.current_chunk_idx = -1

        self.chunk_len = len(labels)
        self.total_len = len(labels) * len(persons)

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        target_chunk_idx = idx // self.chunk_len

        if target_chunk_idx != self.current_chunk_idx:
            # Dynamically load target chunk from disk
            target_person = self.persons[target_chunk_idx]
            self.current_chunk = np.load(f"{self.folder}{target_person}_TI.npy")
            self.current_chunk_idx = target_chunk_idx

        if self.current_chunk is None:
            raise Exception(f"Chunk {target_chunk_idx} is not loaded!")

        relative_idx = idx % self.chunk_len

        x = self.current_chunk[relative_idx]
        y = self.labels[target_chunk_idx][relative_idx]
        return x, y

In [93]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    NumpyDataset(folder="data/out/", persons=train_persons, labels=labels),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)
test_loader = DataLoader(
    NumpyDataset(folder="data/out/", persons=test_persons, labels=labels),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

In [94]:
import torch
import torch.nn as nn

opt = torch.optim.AdamW(model.parameters())
criterion = nn.CrossEntropyLoss()

In [ ]:
import gc
import os

import torch
from torch.amp.autocast_mode import autocast
from tqdm import tqdm

epochs = 20

autocast_dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float16


for epoch in range(epochs):
    model.train()

    pbar = tqdm(train_loader)
    running_loss, correct, seen = 0.0, 0, 0
    for x, y in pbar:
        torch.cuda.empty_cache()
        gc.collect()

        x, y = x.float().to(DEVICE, non_blocking=True), y.float().to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with autocast(DEVICE, autocast_dtype):  # mixed precision
            logits = model(x)
            loss = criterion(logits, y)

        loss.backward()
        opt.step()

        running_loss += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        seen += x.size(0)
        pbar.set_description(f"e{epoch + 1}/{epochs} (train) loss {running_loss / seen:.4f} acc {correct / seen:.3f}")

    model.eval()
    pbar = tqdm(test_loader)
    correct, seen = 0, 0
    for x, y in pbar:
        torch.cuda.empty_cache()
        gc.collect()

        x, y = x.float().to(DEVICE, non_blocking=True), y.float().to(DEVICE, non_blocking=True)

        logits = model(x)

        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        seen += x.size(0)
        pbar.set_description(f"e{epoch + 1}/{epochs} (eval) acc {correct / seen:.3f}")


# ───────────────────── 7.  Save checkpoint ───────────────────────────────
os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/res18_EEG_spectr.pt")
print("✓ training done, weights saved to checkpoints/res18_EEG_spectr.pt")

e3/3 (train) loss 0.3646 acc 0.529:   4%|▍         | 17/396 [00:09<03:36,  1.75it/s]


KeyboardInterrupt: 